# 🎬 HunyuanVideo — Text-to-Video

**Model:** [tencent/HunyuanVideo](https://huggingface.co/tencent/HunyuanVideo)  
**Parametre:** 13B  
**Mimari:** Dual-stream to Single-stream DiT + 3D Causal VAE  
**Çözünürlük:** 720P @ 24fps  
**VRAM:** ~45GB (fp16), ~24GB (offload ile)  

Tencent'in geliştirdiği HunyuanVideo, profesyonel insan değerlendirmelerinde bazı ticari alternatifleri geride bırakmıştır. Görsel kalite ve hareket tutarlılığı konusunda state-of-the-art seviyededir.

In [ ]:
# !pip install torch torchvision diffusers transformers accelerate sentencepiece imageio[ffmpeg]

In [ ]:
import torch
from diffusers import HunyuanVideoPipeline, HunyuanVideoTransformer3DModel
from diffusers.utils import export_to_video

MODEL_ID = "tencent/HunyuanVideo"

transformer = HunyuanVideoTransformer3DModel.from_pretrained(
    MODEL_ID,
    subfolder="transformer",
    torch_dtype=torch.bfloat16,
)

pipe = HunyuanVideoPipeline.from_pretrained(
    MODEL_ID,
    transformer=transformer,
    torch_dtype=torch.float16,
)
pipe.vae.enable_tiling()
pipe.enable_model_cpu_offload()

In [ ]:
prompt = "A cat walks on the grass, realistic style. High quality, 4K resolution."

output = pipe(
    prompt=prompt,
    num_frames=61,
    height=480,
    width=848,
    num_inference_steps=50,
    guidance_scale=6.0,
    generator=torch.Generator(device="cpu").manual_seed(42),
)

export_to_video(output.frames[0], "hunyuan_video_output.mp4", fps=15)
print("Video kaydedildi: hunyuan_video_output.mp4")

In [ ]:
from IPython.display import Video
Video("hunyuan_video_output.mp4", embed=True)